# Representation Bottleneck

Information bottleneck analysis of the residual stream: compression, capacity, redundancy, flow bottlenecks, and cross-position analysis.

**References:**
- Shwartz-Ziv & Tishby (2017) "Opening the Black Box of Deep Neural Networks via Information"
- Saxe et al. (2019) "On the Information Bottleneck Theory of Deep Learning"

## Why This Matters

Representation bottleneck analysis identifies where the network compresses information — where capacity is limited and what information is lost. Compression metrics, capacity estimates, and flow bottleneck detection reveal the information-theoretic constraints on model computation.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.representation_bottleneck import (
    layer_compression_analysis,
    representational_capacity,
    redundancy_analysis,
    information_flow_bottleneck,
    cross_position_redundancy,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

rng = np.random.RandomState(42)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
tokens_list = [jnp.array(rng.randint(0, 100, size=8)) for _ in range(10)]
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

In [ ]:
# 1. Layer compression analysis
comp = layer_compression_analysis(model, tokens_list)
print(f"Bottleneck layer: {comp['bottleneck_layer']}")
print(f"Total compression: {comp['total_compression']:.4f}")
print(f"\nEffective dimensionality per layer:")
for l in range(len(comp['effective_dims'])):
    bar = '#' * int(comp['effective_dims'][l])
    print(f"  Layer {l}: {comp['effective_dims'][l]:.2f} {bar}")

In [ ]:
# 2. Representational capacity
cap = representational_capacity(model, tokens)
print(f"Most utilized layer: {cap['most_utilized_layer']}")
print(f"Least utilized layer: {cap['least_utilized_layer']}")
print(f"\nCapacity per layer:")
for l in range(len(cap['utilization'])):
    print(f"  Layer {l}: util={cap['utilization'][l]:.3f}, top_sv={cap['top_sv_fraction'][l]:.3f}, bits={cap['capacity_bits'][l]:.2f}")

In [ ]:
# 3. Redundancy analysis
red = redundancy_analysis(model, tokens)
print(f"Most redundant layer: {red['most_redundant_layer']}")
print(f"Most transformative layer: {red['most_transformative_layer']}")
print(f"\nInter-layer analysis:")
for l in range(model.cfg.n_layers):
    print(f"  Layer {l}->{l+1}: cos_sim={red['cosine_similarities'][l]:.4f}, "
          f"delta_norm={red['residual_norms'][l]:.4f}, "
          f"rel_change={red['relative_change'][l]:.4f}")

In [ ]:
# 4. Information flow bottleneck
flow = information_flow_bottleneck(model, tokens)
print(f"Bottleneck layer: {flow['bottleneck_layer']}")
print(f"Dominant pathway: {flow['dominant_pathway']}")
print(f"\nAttn vs MLP throughput:")
for l in range(model.cfg.n_layers):
    a_bar = '#' * int(flow['attn_info_fraction'][l] * 30)
    m_bar = '#' * int(flow['mlp_info_fraction'][l] * 30)
    print(f"  Layer {l}: attn={flow['attn_norms'][l]:.3f} ({flow['attn_info_fraction'][l]:.1%}) {a_bar}")
    print(f"           mlp ={flow['mlp_norms'][l]:.3f} ({flow['mlp_info_fraction'][l]:.1%}) {m_bar}")

In [ ]:
# 5. Cross-position redundancy
xpos = cross_position_redundancy(model, tokens)
print(f"Most redundant layer: {xpos['most_redundant_layer']}")
print(f"Most diverse layer: {xpos['most_diverse_layer']}")
print(f"\nPer-layer:")
for l in range(len(xpos['mean_pairwise_similarity'])):
    print(f"  Layer {l}: mean_sim={xpos['mean_pairwise_similarity'][l]:.4f}, "
          f"pos_rank={xpos['position_effective_rank'][l]:.2f}")